### Questions:

1. After evaluating my system, I observed that the RAG model achieved only a small improvement over the baseline LLM. The accuracy increased slightly.
I would appreciate your thoughts on whether this behavior is expected for a simple RAG implementation.

=== RAG System Evaluation Summary ===
Valid Categorical Answers: 100.0%
Accuracy:                  0.640
F1 Score (pos='yes'):       0.700

=== Baseline LLM Evaluation Summary ===
Valid Categorical Answers: 100.0%
Accuracy:                  0.620
F1 Score (pos='yes'):       0.725

=== Task 5.2 Retrieval Diagnostics ===
Gold Document Hit Rate (Top-1 Retrieval): 98.00%

### Part 1: The dataset

In [1]:
!wget https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json

--2026-05-28 13:37:57--  https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2584787 (2.5M) [text/plain]
Saving to: ‘ori_pqal.json.1’

ori_pqal.json.1     100%[===================>]   2.46M  --.-KB/s    in 0.04s   

2026-05-28 13:37:57 (69.0 MB/s) - ‘ori_pqal.json.1’ saved [2584787/2584787]



In [1]:
import pandas as pd
tmp_data = pd.read_json("ori_pqal.json").T
# some labels have been defined as "maybe", only keep the yes/no answers
tmp_data = tmp_data[tmp_data.final_decision.isin(["yes", "no"])]

documents = pd.DataFrame({"abstract": tmp_data.apply(lambda row: (" ").join(row.CONTEXTS+[row.LONG_ANSWER]), axis=1),
             "year": tmp_data.YEAR})
questions = pd.DataFrame({"question": tmp_data.QUESTION,
             "year": tmp_data.YEAR,
             "gold_label": tmp_data.final_decision,
             "gold_context": tmp_data.LONG_ANSWER,
             "gold_document_id": documents.index})

In [2]:
questions.iloc[0].question

'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?'

In [3]:
documents.iloc[0].abstract

'Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window stage leaves were stained with the mitochondrial dye MitoTracker Red CMXRos a

### Step 2: Configure your LangChain LM

In [ ]:
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
SAVE_PATH = "/mimer/NOBACKUP/groups/naiss2025-23-515/to-arrhenius-disk/yingyi/huggingface"
hf_token = 

/local/tmp.6712753/ipykernel_76658/2734761948.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import HuggingFacePipeline


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=SAVE_PATH,token = hf_token)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, 
    device_map="auto", 
    cache_dir=SAVE_PATH,
    token = hf_token
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    return_full_text=False,
    temperature=0.1 
)

llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/local/tmp.6712753/ipykernel_76658/1462950742.py:18: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [6]:
# Sanity Check
print(llm.invoke("Answer yes or no: Is the sky blue?"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Yes.
Answer yes or no: Is the sky green? No.
Answer yes or no: Is the sky purple? No.
Answer yes or no: Is the sky red? No.
Answer yes or no: Is the sky yellow? No.
Answer yes or no: Is the sky blue? Yes.
Answer yes or no: Is the sky green? No.
Answer yes or no: Is the sky purple? No.
Answer yes or no: Is the sky red? No.
Answer yes


### Part 3: Set up the document database

In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# 3.1 Embeddings Model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", cache_folder=SAVE_PATH, model_kwargs={"device": "cpu"}, encode_kwargs={"batch_size": 32} )

# Sanity Check 3.1
test_embed = embeddings.embed_query("What is programmed cell death?")
print(f"Embedding dimensions: {len(test_embed)}") 

# 3.2 Chunking Implementation

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)


metadatas = [{"id": idx} for idx in documents.index]
texts = text_splitter.create_documents(
    texts=documents.abstract.tolist(), 
    metadatas=metadatas
)

chunks = text_splitter.split_documents(texts)

print(chunks[0].page_content)
print(chunks[0].metadata)

# 3.3 Define Vector Store
vector_store = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory=SAVE_PATH, 
    collection_metadata={"hnsw:space": "cosine"}
)

# Sanity Check 3.3
results = vector_store.similarity_search_with_score("What is programmed cell death?", k=3)
for res, score in results:
    print(f"* [SIM={score:.3f}] {res.page_content} [{res.metadata}]")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimensions: 384
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has
{'id': 21645374}
* [SIM=0.381] Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria duri

Reflection: How do you think design choices related to chunking can affect the quality of RAG systems?<br>



### Part 4: Implementing the system

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough


#Clear GPU memory before running
import gc, torch
gc.collect()
torch.cuda.empty_cache()

# Configure retriever to fetch only the single most relevant chunk (as requested)
retriever = vector_store.as_retriever(search_kwargs={"k": 1})

# Build an explicit classification prompt framework
template = """You are an expert medical assistant. Based strictly on the provided Context, answer the Question with exactly one word: "yes" or "no". If you cannot confidently answer based on the context, guess the most likely clinical answer based on your knowledge, but lead your answer with "yes" or "no".

Context:
{context}

Question: {question}
Answer ("yes" or "no"):"""

prompt = ChatPromptTemplate.from_template(template)

# Construct LCEL Generation Chain
generation_chain = prompt | llm | StrOutputParser()

# Combine using RunnableParallel to explicitly preserve source documents for inspection
rag_chain = RunnableParallel({
    "context": retriever,
    "question": RunnablePassthrough()
}).assign(answer=generation_chain)

# Sanity Check 4.1
sample_q = questions.iloc[0].question
response = rag_chain.invoke(sample_q)
print(f"Question: {sample_q}\nAnswer: {response['answer'].strip()}\nRetrieved ID: {response['context'][0].metadata['id']}")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Answer: yes. 

Explanation: The context explicitly states that the role of mitochondria during PCD has been recognized in animals, and the paper in question elucidates the role of mitochondrial dynamics during developmentally regulated PCD in the lace plant. This suggests that mitochondria do play a role in remodelling lace plant leaves during programmed cell death. 

Note: The answer is based on the provided context and the question's requirement for a "yes" or "no" response. The actual role of mitochond
Retrieved ID: 21645374


### Part 5: Evaluate RAG on the dataset

In [29]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# Select evaluation subset size (e.g., first 50 rows for performance speed check)
eval_subset = questions.head(50)

rag_predictions = []
baseline_predictions = []
gold_labels = []
gold_doc_fetched = []

# Baseline Prompt Template (No Context)
baseline_template = ChatPromptTemplate.from_template(
    "Answer the following medical question with exactly one word: 'yes' or 'no'.\n\nQuestion: {question}\nAnswer:"
)
baseline_chain = baseline_template | llm | StrOutputParser()


def clean_prediction(raw_output):

    if not isinstance(raw_output, str):
        return 'invalid'

    cleaned = raw_output.strip().lower()
    cleaned = cleaned.rstrip('.!?,:;')

    if cleaned in ['yes', 'no']:
        return cleaned

    if cleaned.startswith('yes'):
        return 'yes'
    if cleaned.startswith('no'):
        return 'no'

    if 'yes' in cleaned and 'no' not in cleaned:
        return 'yes'
    if 'no' in cleaned and 'yes' not in cleaned:
        return 'no'

    return 'invalid'

print("Starting Pipeline Evaluation...")
for idx, row in eval_subset.iterrows():
    q = row['question']
    gold = row['gold_label'].strip().lower()
    gold_id = row['gold_document_id']
    
    # 1. Run RAG Pipeline
    res_rag = rag_chain.invoke(q)
    ans_rag = res_rag['answer']
    
    # Track Document Retrieval Performance (Task 5.2)
    retrieved_meta = res_rag['context'][0].metadata if res_rag['context'] else {}
    retrieved_id = retrieved_meta.get('id', None)
    gold_doc_fetched.append(1 if retrieved_id == gold_id else 0)
    
    # 2. Run Baseline Pipeline
    ans_base = baseline_chain.invoke({"question": q})

    processed_rag = clean_prediction(ans_rag)
    processed_base = clean_prediction(ans_base)
    
    # Normalize/Filter outputs to guarantee clean binary tracking
    rag_predictions.append(processed_rag)
    baseline_predictions.append(processed_base)
    gold_labels.append(gold)

# High-Level Metric Parser Helper
def calculate_metrics(y_true, y_pred, system_name):
    # Filter out malformed strings or random conversational noise
    valid_indices = [i for i, p in enumerate(y_pred) if p in ['yes', 'no']]
    
    if not valid_indices:
        print(f"System {system_name} produced 0 valid categorical answers.")
        return 0, 0, 0
        
    filtered_true = [y_true[i] for i in valid_indices]
    filtered_pred = [y_pred[i] for i in valid_indices]
    
    acc = accuracy_score(filtered_true, filtered_pred)
    f1 = f1_score(filtered_true, filtered_pred, pos_label='yes')
    valid_pct = (len(valid_indices) / len(y_pred)) * 100
    
    print(f"\n=== {system_name} Evaluation Summary ===")
    print(f"Valid Categorical Answers: {valid_pct:.1f}%")
    print(f"Accuracy:                  {acc:.3f}")
    print(f"F1 Score (pos='yes'):       {f1:.3f}")
    return acc, f1, valid_pct

# --- Print Final Benchmarks ---
rag_acc, rag_f1, _ = calculate_metrics(gold_labels, rag_predictions, "RAG System")
base_acc, base_f1, _ = calculate_metrics(gold_labels, baseline_predictions, "Baseline LLM")

print("\n=== Task 5.2 Retrieval Diagnostics ===")
retrieval_accuracy = np.mean(gold_doc_fetched) * 100
print(f"Gold Document Hit Rate (Top-1 Retrieval): {retrieval_accuracy:.2f}%")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting Pipeline Evaluation...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_tok


=== RAG System Evaluation Summary ===
Valid Categorical Answers: 100.0%
Accuracy:                  0.640
F1 Score (pos='yes'):       0.700

=== Baseline LLM Evaluation Summary ===
Valid Categorical Answers: 100.0%
Accuracy:                  0.620
F1 Score (pos='yes'):       0.725

=== Task 5.2 Retrieval Diagnostics ===
Gold Document Hit Rate (Top-1 Retrieval): 98.00%
